# Phase 6c Wave 5: Ryu-Takayanagi vs Kaul-Majumdar — Technical Notebook

Companion to `papers/note_rt_ch_bounds/paper_draft.tex` (a short formalization note, not a full paper).

**Lean module:** `lean/SKEFTHawking/RTCasiniHuertaBounds.lean` (7 substantive theorems / 0 sorry / 0 new axioms; verified `propext, Classical.choice, Quot.sound` only).

**Structure (mirrors note §1–§7):**
1. The two external-hypothesis predicates: `H_RT_Formula_Valid` and `H_CasiniHuerta_Bound_Valid`
2. The classical RT vs Kaul-Majumdar comparison
3. Quantitative anchor: gap at canonical reduced area = 2 is exactly $(3/2)\log 2$
4. Knife-edge biconditional `rt_eq_kaulMajumdar_iff_trivial_reduced_area`
5. Substantive cross-bridge `rt_falsified_by_kaul_majumdar`
6. Casini-Huerta saturated witness across 2D CFT central charges
7. Sen log-coefficient gap (witness of non-universality)
8. Lean theorem inventory

All numerical content imports from `src.rt_ch_bounds` — no inline physics redefinition.

## 1. The two external hypotheses

`H_RT_Formula_Valid` (a structure on entropy functions $S_{BH}: \mathbb{R} \times \mathbb{R} \to \mathbb{R}$): the function satisfies $S_{BH}(A, G_N) = A/(4 G_N)$ for all positive $A, G_N$. *No log term.*

`H_CasiniHuerta_Bound_Valid` (a structure on entropy functions $S_{\rm ent}: \mathbb{R} \to \mathbb{R}$, indexed by central charge $c$ and UV cutoff $\epsilon$): the function satisfies $S_{\rm ent}(L) \leq (c/3)\log(L/\epsilon)$ for all $L > \epsilon > 0$. *Naming note*: the universal 2D-CFT bound is from Holzhey–Larsen–Wilczek 1994 and Calabrese–Cardy 2004; Casini–Huerta 2009 is the rigorous free-QFT specialization. The Lean predicate retains the historical `H_CasiniHuerta_*` prefix for compatibility.

Both hypotheses are *external*: the bulk minimal-surface construction (Lewkowycz-Maldacena replica trick) and the universal-2D-CFT proof (modular Hamiltonian / replica trick / direct CFT manipulations) are out of scope for this note.

In [ ]:
import os, sys
_HERE = os.getcwd()
_PROJ = _HERE if os.path.basename(_HERE) == 'SK_EFT_Hawking' else os.path.dirname(_HERE)
if _PROJ not in sys.path:
    sys.path.insert(0, _PROJ)

from src.rt_ch_bounds import (
    classical_rt_entropy,
    kaul_majumdar_entropy,
    rt_kaul_majumdar_gap,
    sen_4d_log_coeff,
    SEN_KAUL_MAJUMDAR_COEFF_GAP,
    saturated_ch_entropy,
    ch_log_bound,
    ch_bound_holds,
    central_charge_from_saturation,
)
from src.core.visualizations import COLORS, fig_rt_ch_bounds_mtc
import math

print('Classical RT (no log term):     S_RT(A, G_N) = A / (4 G_N)')
print('Kaul-Majumdar (microscopic):    S_KM(A, G_N) = A / (4 G_N) - (3/2) log(A / (4 G_N))')
print('Universal 2D-CFT log bound:     S_ent(L) ≤ (c/3) log(L / ε)')
print('  (Holzhey-Larsen-Wilczek 1994, Calabrese-Cardy 2004; Casini-Huerta 2009 = free-QFT case)')

## 2. Classical RT vs Kaul-Majumdar across reduced area

Reduced area $\equiv A/(4 G_N)$. Classical RT is linear in reduced area; Kaul-Majumdar has a $-\frac{3}{2}\log$ correction. Compute both at a sweep of reduced areas to see the structural disagreement.

In [2]:
G_N = 1.0
header = f'  {"reduced area":>14} | {"classical RT":>14} | {"Kaul-Majumdar":>14} | {"gap":>10}'
print(header)
print('  ' + '-' * (len(header) - 2))
for r in [0.5, 1.0, 2.0, 4.0, 10.0, 100.0]:
    A = r * 4 * G_N
    rt = classical_rt_entropy(A, G_N)
    km = kaul_majumdar_entropy(A, G_N, c0=0.0)
    gap = rt_kaul_majumdar_gap(A, G_N)
    print(f'  {r:>14.2f} | {rt:>14.4f} | {km:>14.4f} | {gap:>10.4f}')

print()
print('  → Gap = 0 only at reduced area = 1; growing positively above 1, negatively below 1.')

    reduced area |   classical RT |  Kaul-Majumdar |        gap
  -------------------------------------------------------------
            0.50 |         0.5000 |         1.5397 |    -1.0397
            1.00 |         1.0000 |         1.0000 |     0.0000
            2.00 |         2.0000 |         0.9603 |     1.0397
            4.00 |         4.0000 |         1.9206 |     2.0794
           10.00 |        10.0000 |         6.5461 |     3.4539
          100.00 |       100.0000 |        93.0922 |     6.9078

  → Gap = 0 only at reduced area = 1; growing positively above 1, negatively below 1.


## 3. Quantitative anchor at reduced area = 2

Lean theorem `rt_kaulMajumdar_gap_at_reduced_area_two`:
$$S_{RT}(A=8 G_N, G_N) - S_{KM}(A=8 G_N, G_N) = \tfrac{3}{2} \log 2 \approx 1.040.$$

This is *exact* — not asymptotic, not approximate. The proof unfolds `kaulMajumdarS`, normalizes the arithmetic, and closes with `ring`.

In [3]:
G_N = 1.0
A = 8.0 * G_N
gap_computed = rt_kaul_majumdar_gap(A, G_N)
gap_predicted = 1.5 * math.log(2)
print(f'  Reduced area = 2 → A = 8 G_N (G_N = 1)')
print(f'  Computed gap        = {gap_computed:.10f}')
print(f'  (3/2) log 2         = {gap_predicted:.10f}')
print(f'  match (tol 1e-12)   = {abs(gap_computed - gap_predicted) < 1e-12}')

  Reduced area = 2 → A = 8 G_N (G_N = 1)
  Computed gap        = 1.0397207708
  (3/2) log 2         = 1.0397207708
  match (tol 1e-12)   = True


## 4. Knife-edge biconditional

Lean theorem `rt_eq_kaulMajumdar_iff_trivial_reduced_area`:
$$\frac{A}{4 G_N} = S_{KM}(A, G_N) \;\Longleftrightarrow\; \frac{A}{4 G_N} = 1.$$

Within the *specific entropy functions* used here — classical RT $f(A,G_N) = A/(4 G_N)$ and Kaul-Majumdar $S_{KM}(A,G_N) = A/(4 G_N) - (3/2)\log(A/(4 G_N))$ — the two agree on the unique knife-edge $A = 4 G_N$ (Planckian scale); elsewhere their difference is $(3/2)\log(A/(4 G_N))$ by direct subtraction. The proof of the forward direction extracts $\log = 0$ from the equality (using `Real.exp_log` injectivity); reverse direction uses $\log 1 = 0$.

(The gap formula is a definitional unfolding for these two specific functions, not a general structural theorem about all RT-like and Kaul-Majumdar-like extensions; the *substantive* content is locating the unique agreement point.)

In [4]:
G_N = 1.0
for A in [1.0, 4.0, 4.0 + 1e-9, 8.0]:
    reduced = A / (4 * G_N)
    rt = classical_rt_entropy(A, G_N)
    km = kaul_majumdar_entropy(A, G_N, c0=0.0)
    agree = abs(rt - km) < 1e-12
    print(f'  A = {A:.10f}  reduced = {reduced:.10f}  RT = {rt:.6f}  KM = {km:.6f}  agree: {agree}')
print()
print('  → RT and KM agree iff reduced area = 1, mirroring rt_eq_kaulMajumdar_iff_trivial_reduced_area.')

  A = 1.0000000000  reduced = 0.2500000000  RT = 0.250000  KM = 2.329442  agree: False
  A = 4.0000000000  reduced = 1.0000000000  RT = 1.000000  KM = 1.000000  agree: True
  A = 4.0000000010  reduced = 1.0000000003  RT = 1.000000  KM = 1.000000  agree: False
  A = 8.0000000000  reduced = 2.0000000000  RT = 2.000000  KM = 0.960279  agree: False

  → RT and KM agree iff reduced area = 1, mirroring rt_eq_kaulMajumdar_iff_trivial_reduced_area.


## 5. Substantive cross-bridge: `rt_falsified_by_kaul_majumdar`

Lean: assuming $H_{RT}$ holds for some entropy function $S_{BH}$, then at any non-trivial reduced area, $S_{BH}(A, G_N) \neq S_{KM}(A, G_N, 0)$. Proof: rewrite $S_{BH}$ using $h_{RT}$.rt_proportional, invoke contrapositive of the knife-edge biconditional.

**Direct falsifier** `kaulMajumdar_not_H_RT`: the function $\lambda A\, G_N \mapsto S_{KM}(A, G_N, 0)$ does NOT satisfy $H_{RT}$. Concrete counterexample: $A = 8, G_N = 1$, where $S_{KM} = 2 - (3/2)\log 2 \approx 0.96 \neq 2$. Proof closes via `Real.log 2 > 0` and `linarith`.

In [5]:
A, G_N = 8.0, 1.0
rt_value = A / (4 * G_N)
km_value = kaul_majumdar_entropy(A, G_N, 0)
delta = abs(rt_value - km_value)
print(f'  Concrete witness: A = {A}, G_N = {G_N}')
print(f'    classical RT formula      → {rt_value}')
print(f'    Kaul-Majumdar evaluation  → {km_value:.6f}')
print(f'    |Δ|                        = {delta:.6f}      (= (3/2) log 2 ≈ 1.0397)')
print(f'    > 0 (log 2 > 0): mirrors kaulMajumdar_not_H_RT closing tactic.')

  Concrete witness: A = 8.0, G_N = 1.0
    classical RT formula      → 2.0
    Kaul-Majumdar evaluation  → 0.960279
    |Δ|                        = 1.039721      (= (3/2) log 2 ≈ 1.0397)
    > 0 (log 2 > 0): mirrors kaulMajumdar_not_H_RT closing tactic.


## 6. Casini-Huerta saturated witness across 2D CFTs

The saturated entropy $S_{\rm ent}(L) := (c/3)\log(L/\epsilon)$ exactly meets the CH bound. Verifies the witness `H_CasiniHuerta_Bound_Valid_witness_saturated`. Compute across canonical 2D CFT central charges.

In [6]:
UV = 1e-3
L = 1.0
header = f'  {"CFT":24s} | {"c":>4} | {"S_ent(L=1) saturated":>22} | {"recovers c?":>14}'
print(header)
print('  ' + '-' * (len(header) - 2))
for label, c in [
    ('Ising',                 0.5),
    ('Tricritical Ising',     0.7),
    ('3-state Potts',         0.8),
    ('Free boson',            1.0),
]:
    S = saturated_ch_entropy(L, c, UV)
    c_recovered = central_charge_from_saturation(S, L, UV)
    print(f'  {label:24s} | {c:>4.2f} | {S:>22.6f} | {c_recovered:>14.6f}')

print()
print('  → Saturated entropy meets CH bound exactly; round-trip recovers central charge.')

  CFT                      |    c |   S_ent(L=1) saturated |    recovers c?
  -------------------------------------------------------------------------
  Ising                    | 0.50 |               1.151293 |       0.500000
  Tricritical Ising        | 0.70 |               1.611810 |       0.700000
  3-state Potts            | 0.80 |               1.842068 |       0.800000
  Free boson               | 1.00 |               2.302585 |       1.000000

  → Saturated entropy meets CH bound exactly; round-trip recovers central charge.


## 7. Sen 4D log-coefficient gap

Sen (arXiv:1205.0971, JHEP 2013) computed the log-coefficient for 4D Schwarzschild via string-theory amplitudes: $77/45 \approx 1.711$. This *disagrees* with Kaul-Majumdar's $-3/2 = -1.5$. Both are microscopic-counting answers; both are claimed universal within their derivation. The gap is exactly:
$$\frac{77}{45} - \left(-\frac{3}{2}\right) = \frac{154 + 135}{90} = \frac{289}{90} \approx 3.211.$$
Witness that log-coefficient *universality* across regularizations is FALSE — different microscopic theories give different log coefficients.

In [7]:
print(f'  Sen 4D Schwarzschild log coefficient   = {sen_4d_log_coeff():.6f}    (= 77/45 = 212/45 - 3)')
print(f'  Kaul-Majumdar log coefficient          = {-1.5:.6f}    (= -3/2)')
print(f'  Gap (Sen − Kaul-Majumdar)              = {SEN_KAUL_MAJUMDAR_COEFF_GAP:.6f}    (= 289/90)')
print(f'  Predicted gap from formula             = {77/45 - (-3/2):.6f}')
print()
print('  → Lean theorem (in BHEntropyMicroscopic.lean): sen_4d_disagrees_with_kaul_majumdar')
print('  → Establishes log-coefficient NON-universality across microscopic regularizations.')

  Sen 4D Schwarzschild log coefficient   = 1.711111    (= 77/45 = 212/45 - 3)
  Kaul-Majumdar log coefficient          = -1.500000    (= -3/2)
  Gap (Sen − Kaul-Majumdar)              = 3.211111    (= 289/90)
  Predicted gap from formula             = 3.211111

  → Lean theorem (in BHEntropyMicroscopic.lean): sen_4d_disagrees_with_kaul_majumdar
  → Establishes log-coefficient NON-universality across microscopic regularizations.


## 8. Figure — RT/KM crossing at the knife-edge + CH saturation across CFTs

**Left panel.** Classical RT (linear, steel-blue solid) vs W3 Kaul-Majumdar (dashed amber) across reduced area $A/(4 G_N) \in [0.1, 5.0]$. The two curves cross at exactly reduced area = 1 — the knife-edge. Below 1, KM > RT (negative log correction lifts KM above the linear); above 1, RT > KM.

**Right panel.** Saturated CH entropy $(c/3)\log(L/\epsilon)$ across canonical 2D CFTs (Ising c = 1/2, tricritical Ising c = 7/10, 3-state Potts c = 4/5, free boson c = 1), plotted on a log-$L$ axis. All four are linear on the semilog plot, separated by central charge.

In [8]:
# viz-ref: fig_rt_ch_bounds_mtc
fig_rt_ch_bounds_mtc().show()

## 9. Lean theorem inventory (7 substantive theorems + 2 tracked-Prop structures)

The seven substantive theorems and two tracked-Prop structures in `RTCasiniHuertaBounds.lean` are clean (zero `sorry`, zero new axioms; closure $\{\texttt{propext, Classical.choice, Quot.sound}\}$). counts.tex `\rtChThms{}` reports the substantive-theorem count = 7; the table below additionally lists the two `H_` structures for completeness.

| # | Theorem / Definition | Role |
|---|----------------------|------|
| — | `H_RT_Formula_Valid` (structure / tracked Prop) | tracked external Prop: $S_{BH} = A/(4 G_N)$ on positive cone |
| — | `H_CasiniHuerta_Bound_Valid` (structure / tracked Prop) | tracked external Prop: $S_{\rm ent} \leq (c/3)\log(L/\epsilon)$ |
| 1 | `rt_entropy_pos` | RT entropy strictly positive |
| 2 | `rt_kaulMajumdar_gap_at_reduced_area_two` | quantitative anchor: gap = $(3/2)\log 2$ |
| 3 | `rt_eq_kaulMajumdar_iff_trivial_reduced_area` | knife-edge biconditional (named correctness-push) |
| 4 | `rt_falsified_by_kaul_majumdar` | substantive cross-bridge |
| 5 | `H_CasiniHuerta_Bound_Valid_witness_saturated` | concrete witness at $S(L) := (c/3)\log(L/\epsilon)$ |
| 6 | `kaulMajumdar_not_H_RT` | direct falsifier |
| 7 | `ch_log_bound_pos_at_log_pos` | CH bound is positive whenever $L > \epsilon$ |

Cross-wave strengthening (2026-04-28-0030) removed `rt_classical_inconsistent_with_kaul_majumdar` — content was P2-redundant with the knife-edge biconditional contrapositive; the cross-bridge `rt_falsified_by_kaul_majumdar` was rerouted to invoke the biconditional directly.

**Out of scope (per Phase 6c roadmap §A):** bulk minimal-surface construction (Lewkowycz-Maldacena replica trick), full CH derivation from modular Hamiltonian, AdS/CFT spectrum identification.